<a href="https://colab.research.google.com/github/nickcanoy/Masters_in_DataScience/blob/main/Data_Preparation_CANOY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#RODENICK CANOY | Data Preparation | PSMDSSC 101-PSMDS12G2 - Data Mining

In [ ]:
# 1. INSTALL & IMPORT REQUIRED LIBRARIES

import pandas as pd
import numpy as np

In [ ]:
# 2. LOAD THE DATASET

from google.colab import files
uploaded = files.upload()

Saving hospitalizations-1.csv to hospitalizations-1 (1).csv


In [ ]:
# Load CSV
df = pd.read_csv("hospitalizations-1.csv")

In [ ]:
# 3. PREVIEW THE DATASET
print("Dataset Preview:")
df.head()

Dataset Preview:


,date,location_key,new_hospitalized_patients,cumulative_hospitalized_patients,current_hospitalized_patients,new_intensive_care_patients,cumulative_intensive_care_patients,current_intensive_care_patients,new_ventilator_patients,cumulative_ventilator_patients,current_ventilator_patients
0,0022-01-10,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN
1,0022-01-20,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN
2,0202-03-30,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN
3,0221-07-06,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN
4,1202-01-07,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN


In [ ]:
# Shape, column names, data types
print("Dataset Shape:", df.shape)
print("\nColumn Info:")
df.info()

print("\nMissing Values:")
df.isna().sum()

Dataset Shape: (1768485, 11)

Column Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1768485 entries, 0 to 1768484
Data columns (total 11 columns):
 #   Column                              Dtype  
---  ------                              -----  
 0   date                                object 
 1   location_key                        object 
 2   new_hospitalized_patients           float64
 3   cumulative_hospitalized_patients    float64
 4   current_hospitalized_patients       float64
 5   new_intensive_care_patients         float64
 6   cumulative_intensive_care_patients  float64
 7   current_intensive_care_patients     float64
 8   new_ventilator_patients             float64
 9   cumulative_ventilator_patients      float64
 10  current_ventilator_patients         float64
dtypes: float64(9), object(2)
memory usage: 148.4+ MB

Missing Values:


,0
date,0
location_key,0
new_hospitalized_patients,119333
cumulative_hospitalized_patients,123111
current_hospitalized_patients,1584627
new_intensive_care_patients,620623
cumulative_intensive_care_patients,622435
current_intensive_care_patients,1576562
new_ventilator_patients,1757509
cumulative_ventilator_patients,1760209


In [ ]:
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

After removing duplicates: (1768485, 11)


In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.head()

,date,location_key,new_hospitalized_patients,cumulative_hospitalized_patients,current_hospitalized_patients,new_intensive_care_patients,cumulative_intensive_care_patients,current_intensive_care_patients,new_ventilator_patients,cumulative_ventilator_patients,current_ventilator_patients
0,0022-01-10,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN
1,0022-01-20,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN
2,0202-03-30,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN
3,0221-07-06,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN
4,1202-01-07,AR,0.0,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN


In [ ]:
# Tries to parse date even if format varies
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df['date'].isna().sum()   # Check if any date failed to convert

/tmp/ipython-input-241270319.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


np.int64(27)

In [ ]:
# Identify numeric columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Option A: Fill NA numeric values with 0
df[numeric_cols] = df[numeric_cols].fillna(0)

In [ ]:
# Basic Z-score outlier identification
from scipy.stats import zscore

z_scores = np.abs(zscore(df[numeric_cols]))
outliers = np.where(z_scores > 3)
print("Number of potential outliers:", len(outliers[0]))

Number of potential outliers: 36997


In [ ]:
# ============================================
# STEP 6 — Integrate Hospitalization + Vaccination
# ============================================

# --- 1. Standardize keys ---
hospital['location_key'] = hospital['location_key'].astype(str).str.upper().str.strip()
vaccination['location_key'] = vaccination['location_key'].astype(str).str.upper().str.strip()

# Convert date fields to datetime if needed
hospital['date'] = pd.to_datetime(hospital['date'], errors='coerce')
vaccination['date'] = pd.to_datetime(vaccination['date'], errors='coerce')

# --- 2. Select relevant vaccination columns ---
vacc_core_cols = [
    'location_key', 'date',
    'new_persons_vaccinated',
    'cumulative_persons_vaccinated',
    'new_persons_fully_vaccinated',
    'cumulative_persons_fully_vaccinated',
    'new_vaccine_doses_administered',
    'cumulative_vaccine_doses_administered'
]

vacc_clean = vaccination[vacc_core_cols].copy()

# --- 3. Merge datasets by location_key + date ---
integrated_df = hospital.merge(
    vacc_clean,
    on=['location_key', 'date'],
    how='left'
)

# --- 4. Fill missing vaccination values ---
vacc_numeric_cols = [
    'new_persons_vaccinated',
    'cumulative_persons_vaccinated',
    'new_persons_fully_vaccinated',
    'cumulative_persons_fully_vaccinated',
    'new_vaccine_doses_administered',
    'cumulative_vaccine_doses_administered'
]

# Modify column names to match the suffixed columns in integrated_df after merge
vacc_numeric_cols_suffixed = [col + '_y' for col in vacc_numeric_cols]
integrated_df[vacc_numeric_cols_suffixed] = integrated_df[vacc_numeric_cols_suffixed].fillna(0)

# --- 5. View and save output ---
integrated_df.head()
integrated_df.to_csv("hospital_vaccination_merged.csv", index=False)

print("✔ Integration complete! File saved as hospital_vaccination_merged.csv")

✔ Integration complete! File saved as hospital_vaccination_merged.csv
